colours same as paper, Y-axis for both SS and FS should end at the same And also end at 25. Font is Arial, font size is 12, and it's in bold. Finish the x-axis at 10 minutes. Add a hyphenated x-axis to make it clear what the bin span is. 

In [3]:
"""
Builds a time-binned version of the approach/baseline speed classification
already used in the parametric-sweep script (compute_approach_speeds_from_file),
so it can feed into a time-trend analysis (time_trend_analysis_speed.py) the
same way advanced_orientation_analysis.py's compute_time_course() feeds
time_trend_analysis_orientation.py.

WHY THIS EXISTS: approach_speed_summary_all_radii.csv has ONE approach-speed
and ONE baseline-speed value per video per stimulus per radius -- a
whole-session average, with no time axis. To ask "does approach speed change
over the course of the session" we need the same per-frame approach/baseline
classification, but preserved at time-bin resolution instead of collapsed
to a single number.

The frame classification logic (radial closing velocity threshold, minimum
body-speed gate, interaction-radius exclusion) is based on
compute_approach_speeds_from_file() in the parametric-sweep script -- NOT
reimplemented from scratch -- so the numbers here stay consistent with
whatever that script already reports for the same radius, EXCEPT for one
deliberate addition: a frame now also has to satisfy an orientation
criterion (abs_angle_to_SS/FS <= ANGLE_THRESHOLD_DEG) to count as
"approach". The original distance-only definition couldn't distinguish a
genuine oriented approach from the animal merely closing distance while
moving sideways or backward past the stimulus; this ANDs a facing
requirement on top of the existing near/closing/moving criteria. Masks are
computed and kept at the frame level (compute_speed_masks), then averaged
within time bins (compute_speed_time_course), instead of being averaged
once across the whole session.

INTERACTION_RADIUS_CM is fixed at 6 cm per the cutoff already chosen for the
paired approach-vs-baseline analysis, so time-trend results here line up
with that plot. ANGLE_THRESHOLD_DEG is fixed at 45 deg, matching the
mid-point of the parametric sweep's own ANGLE_THRESHOLDS (30-60 deg).

THREE STAGES IN THIS SCRIPT
---------------------------
1. build_speed_time_course()      -> time_course_speed.csv
                                     (one row per video per time bin)
2. compute_fit_quality()          -> time_course_speed_fit_quality.csv
                                     (per-video R^2, slope, p for the
                                      within-video time trend)
3. analyze_treatment_trends()     -> time_course_speed_treatment_trends.csv
                                     (GEE + mixed-model treatment x time
                                      interaction tests, robust SEs, global
                                      Wald test across contrasts, simple
                                      slopes per group, Holm correction)

REFERENCE GROUP: GH AL is fixed as the control/reference group for the
treatment-trend stage (REFERENCE_GROUP = "GH AL"). All interaction
coefficients are therefore contrasts of each other group *against GH AL*.
"""

import os
import re
import numpy as np
import pandas as pd
from glob import glob
from scipy import stats
from itertools import combinations

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
except ImportError as e:
    raise ImportError(
        "This script needs statsmodels for the treatment-trend stage. "
        "Install with: pip install statsmodels"
    ) from e

# =============================================================================
# CONFIG -- kept identical to the parametric-sweep script's speed-related
# settings, so results at RADIUS=6 match what that script already reports.
# =============================================================================
MASTER_SHEET_PATH = "housing_food_condition.xlsx"
AUGMENTED_CSV_FOLDER = "all_augmented_csv/results_with_vertices"
OUTPUT_FOLDER = "advanced_analysis_plots"   # same output folder as the other time-trend scripts

FPS = 25
APPROACH_THRESHOLD = 0.5          # min radial closing velocity (cm/s) -- same as parametric sweep
MIN_SPEED_FOR_APPROACH = 1.0      # min overall body speed (cm/s) -- same as parametric sweep
DISTANCE_SMOOTH_WINDOW = 5        # same as parametric sweep
INTERACTION_RADIUS_CM = 6.0       # fixed cutoff (chosen for the approach-vs-baseline analysis)
ANGLE_THRESHOLD_DEG = 45.0        # max abs_angle_to_SS/FS (deg) to count as "facing" -- ANDed
                                   # with the distance/velocity/speed criteria below, so a frame
                                   # only counts as "approach" if the animal is near, closing in,
                                   # moving, AND actually oriented toward the stimulus (not just
                                   # moving toward it sideways/backward). Matches the mid-point of
                                   # the parametric sweep's ANGLE_THRESHOLDS (30-60 deg) for
                                   # consistency with the facing-time analysis.
TIME_BIN_SECONDS = 60             # same default bin width as the orientation time-course

GROUP_COL_CANDIDATES = ['Treatment', 'Housing', 'Food']

# Minimum number of bins with non-NaN approach speed needed to attempt a
# per-video linear fit. linregress will happily fit 2 points, but R^2 from 2
# points is meaningless.
MIN_BINS_FOR_FIT = 3

# ---- Treatment-trend stage config ----
TREATMENT_COL = 'Treatment'       # which grouping column to use for the GEE/MM stage.
                                   # Set to None to auto-pick the first available
                                   # column from GROUP_COL_CANDIDATES.
OUTCOMES_FOR_TREND = ['approach_speed_SS', 'approach_speed_FS']
WORKING_CORR = 'ar'                # 'ar' (AR(1)), 'exchangeable', 'independence'
FAMILY = sm.families.Gaussian()    # identity link; continuous outcome
REFERENCE_GROUP = "GH AL"          # <-- GH AL is the fixed control/reference group
APPLY_HOLM = True                  # Holm-correct the 3 interaction p-values within each outcome
SMALL_SAMPLE_T_CORRECTION = True   # use t(df = n_clusters - n_params) instead of z for GEE p-values
CORR_STRUCT_CHOICES = ['ar', 'exchangeable', 'independence']  # for the QIC-like sensitivity loop


# =============================================================================
# Helpers (file/video matching, master sheet -- same approach as the other
# advanced-analysis scripts, duplicated here so this script is self-contained)
# =============================================================================
def get_base_name_from_file(csv_path):
    base = os.path.basename(csv_path)
    if base.startswith("aug_"):
        base = base[4:]
    if base.endswith(".csv"):
        base = base[:-4]
    return base


def extract_video_name_from_filename(basename, master_videos):
    matches = [mv for mv in master_videos if basename.startswith(mv)]
    if not matches:
        return None
    return max(matches, key=len)


def load_master_sheet(master_path):
    master = pd.read_excel(master_path, sheet_name=0)
    master = master.dropna(subset=['Video'])

    group_cols_available = []
    for col in GROUP_COL_CANDIDATES:
        if col in master.columns:
            group_cols_available.append(col)
        else:
            print(f"  [INFO] Column '{col}' not found in master sheet ({master_path}); "
                  f"skipping this grouping variable.")

    if not group_cols_available:
        raise ValueError(
            f"Master sheet must contain at least one of {GROUP_COL_CANDIDATES}. "
            f"Found columns: {list(master.columns)}"
        )

    keep_cols = ['Video'] + group_cols_available
    master = master[keep_cols].copy()
    master.rename(columns={'Video': 'video'}, inplace=True)
    print(f"  Grouping variables that will be tested: {group_cols_available}")
    return master, group_cols_available


# =============================================================================
# Frame-level classification -- copied verbatim (structurally) from
# compute_approach_speeds_from_file() in the parametric-sweep script, except
# it returns the frame-level masks/speed array instead of collapsing them to
# per-session means, so time-binning can happen afterward.
# =============================================================================
def compute_speed_masks(df, fps=FPS, threshold=APPROACH_THRESHOLD,
                         min_speed_for_approach=MIN_SPEED_FOR_APPROACH,
                         interaction_radius_cm=INTERACTION_RADIUS_CM,
                         distance_smooth_window=DISTANCE_SMOOTH_WINDOW,
                         angle_threshold_deg=ANGLE_THRESHOLD_DEG):
    """
    "Approach" now requires the animal to be oriented toward the stimulus,
    not just closing distance on it: a frame only counts as approach_ss if
    it satisfies the original near/closing/moving criteria AND
    abs_angle_to_SS <= angle_threshold_deg (facing the stimulus). This
    excludes frames where the animal is moving toward the stimulus
    sideways or backward, which the distance-only version couldn't tell
    apart from a genuine oriented approach.
    """
    required = ['dist_to_SS_cm', 'dist_to_FS_cm', 'in_SC', 'in_FC', 'speed_cm_per_s',
                'abs_angle_to_SS', 'abs_angle_to_FS']
    if not all(c in df.columns for c in required):
        return None

    dt = 1.0 / fps
    dist_ss = df['dist_to_SS_cm'].values
    dist_fs = df['dist_to_FS_cm'].values

    if distance_smooth_window and distance_smooth_window > 1:
        dist_ss_smooth = pd.Series(dist_ss).rolling(
            distance_smooth_window, center=True, min_periods=1).median().values
        dist_fs_smooth = pd.Series(dist_fs).rolling(
            distance_smooth_window, center=True, min_periods=1).median().values
    else:
        dist_ss_smooth, dist_fs_smooth = dist_ss, dist_fs

    rad_vel_ss = -np.gradient(dist_ss_smooth, dt)
    rad_vel_fs = -np.gradient(dist_fs_smooth, dt)
    in_sc = df['in_SC'].values
    in_fc = df['in_FC'].values
    speed = df['speed_cm_per_s'].values

    angle_ss = df['abs_angle_to_SS'].values
    angle_fs = df['abs_angle_to_FS'].values
    facing_ss = (angle_ss <= angle_threshold_deg) & (~np.isnan(angle_ss))
    facing_fs = (angle_fs <= angle_threshold_deg) & (~np.isnan(angle_fs))

    interacting_ss = in_sc & (dist_ss <= interaction_radius_cm) & (~np.isnan(dist_ss))
    interacting_fs = in_fc & (dist_fs <= interaction_radius_cm) & (~np.isnan(dist_fs))

    approach_ss = (in_sc & (rad_vel_ss > threshold) & (speed > min_speed_for_approach)
                   & facing_ss & (~interacting_ss) & (~np.isnan(dist_ss)))
    approach_fs = (in_fc & (rad_vel_fs > threshold) & (speed > min_speed_for_approach)
                   & facing_fs & (~interacting_fs) & (~np.isnan(dist_fs)))

    baseline_ss = in_sc & (~approach_ss) & (~interacting_ss) & (~np.isnan(dist_ss))
    baseline_fs = in_fc & (~approach_fs) & (~interacting_fs) & (~np.isnan(dist_fs))

    return {
        'speed': speed,
        'approach_ss': approach_ss,
        'baseline_ss': baseline_ss,
        'approach_fs': approach_fs,
        'baseline_fs': baseline_fs,
    }


def compute_speed_time_course(masks, video_name, fps=FPS, bin_seconds=TIME_BIN_SECONDS):
    """
    Bins the session into bin_seconds-wide windows and computes, per bin, the
    mean speed within each of the four frame classes (approach/baseline x
    SS/FS). A bin with zero frames of a given class gets NaN for that
    column, not zero -- e.g. "no approach frames this minute" should not be
    read as "approach speed was 0 cm/s this minute".
    """
    speed = masks['speed']
    n = len(speed)
    bin_size_frames = max(1, int(round(bin_seconds * fps)))
    time_bin = (np.arange(n) // bin_size_frames).astype(int)

    rows = []
    for b in np.unique(time_bin):
        idx = (time_bin == b)

        def masked_mean(mask):
            sel = mask & idx
            return np.nanmean(speed[sel]) if sel.any() else np.nan

        rows.append({
            'video': video_name,
            'time_bin': int(b),
            'bin_start_s': b * bin_seconds,
            'bin_start_min': b * bin_seconds / 60.0,
            'approach_speed_SS': masked_mean(masks['approach_ss']),
            'baseline_speed_SS': masked_mean(masks['baseline_ss']),
            'approach_speed_FS': masked_mean(masks['approach_fs']),
            'baseline_speed_FS': masked_mean(masks['baseline_fs']),
            'n_frames': int(idx.sum()),
        })
    return pd.DataFrame(rows)


# =============================================================================
# Per-video fit-quality metrics (R^2, slope, p) for the within-video time trend
# =============================================================================
def compute_fit_quality(time_course_df, group_cols, min_bins=MIN_BINS_FOR_FIT):
    """
    For each video, fit a straight line to approach_speed_SS vs time_bin and
    to approach_speed_FS vs time_bin, and report slope, intercept, R^2,
    p-value, and N bins. Per-video (rather than pooled) fitting is deliberate:
    pooling would let between-animal differences dominate R^2.
    """
    outcomes = ['approach_speed_SS', 'approach_speed_FS']
    fit_rows = []

    for vname, g in time_course_df.groupby('video'):
        group_values = {gc: g[gc].iloc[0] for gc in group_cols}
        for outcome in outcomes:
            sub = g[['time_bin', outcome]].dropna()
            n = len(sub)
            if n < min_bins:
                fit_rows.append({
                    'video': vname, **group_values, 'outcome': outcome,
                    'slope': np.nan, 'intercept': np.nan,
                    'r_value': np.nan, 'r_squared': np.nan,
                    'p_value': np.nan, 'std_err': np.nan,
                    'n_bins': n, 'note': f'<{min_bins} usable bins'
                })
                continue

            x = sub['time_bin'].values.astype(float)
            y = sub[outcome].values.astype(float)

            if np.allclose(y, y[0]) or np.ptp(x) == 0:
                fit_rows.append({
                    'video': vname, **group_values, 'outcome': outcome,
                    'slope': 0.0, 'intercept': float(y[0]),
                    'r_value': 0.0, 'r_squared': 0.0,
                    'p_value': np.nan, 'std_err': np.nan,
                    'n_bins': n, 'note': 'zero variance in y'
                })
                continue

            lr = stats.linregress(x, y)
            fit_rows.append({
                'video': vname, **group_values, 'outcome': outcome,
                'slope': lr.slope,
                'intercept': lr.intercept,
                'r_value': lr.rvalue,
                'r_squared': lr.rvalue ** 2,
                'p_value': lr.pvalue,
                'std_err': lr.stderr,
                'n_bins': n,
                'note': ''
            })

    return pd.DataFrame(fit_rows)


def print_fit_summary(fit_df):
    print("\n" + "=" * 70)
    print("PER-VIDEO FIT SUMMARY (approach speed ~ time_bin)")
    print("=" * 70)
    for outcome in ['approach_speed_SS', 'approach_speed_FS']:
        sub = fit_df[fit_df['outcome'] == outcome].dropna(subset=['r_squared'])
        if sub.empty:
            print(f"\n{outcome}: no videos had enough usable bins to fit.")
            continue
        print(f"\n{outcome}:  n_videos = {len(sub)}")
        print(f"  median R^2   = {sub['r_squared'].median():.3f}")
        print(f"  mean   R^2   = {sub['r_squared'].mean():.3f}")
        print(f"  range  R^2   = {sub['r_squared'].min():.3f} .. {sub['r_squared'].max():.3f}")
        print(f"  n videos p<0.05 = {(sub['p_value'] < 0.05).sum()} / {len(sub)}")
        print(f"  median slope = {sub['slope'].median():.4f} cm/s per bin")


# =============================================================================
# NEW STAGE: treatment x time trend tests (GEE primary, mixed model sensitivity)
# =============================================================================
def _holm_correction(pvals):
    """Holm-Bonferroni step-down correction. Returns adjusted p-values in the
    same order as the input."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(n, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = (n - rank) * pvals[idx]
        running_max = max(running_max, val)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def _fit_gee(df, formula, groups_col, cov_struct_name, family):
    """
    Fit a GEE with robust (sandwich) SEs. Returns the fitted result.
    cov_struct_name: one of 'ar', 'exchangeable', 'independence'.
    """
    cov_map = {
        'ar': sm.cov_struct.Autoregressive,
        'exchangeable': sm.cov_struct.Exchangeable,
        'independence': sm.cov_struct.Independence,
    }
    if cov_struct_name not in cov_map:
        raise ValueError(f"Unknown cov_struct '{cov_struct_name}'. "
                         f"Choose from {list(cov_map)}.")
    model = smf.gee(
        formula=formula,
        groups=groups_col,
        data=df,
        family=family,
        cov_struct=cov_map[cov_struct_name](),
    )
    return model.fit(cov_type='robust')


def _fit_mixedlm(df, formula, groups_col):
    """
    Fit a linear mixed model with random intercept + random slope on time_bin
    per animal. REML=False so we get a likelihood for AIC/BIC.
    """
    model = smf.mixedlm(
        formula=formula,
        data=df,
        groups=df[groups_col],
        re_formula="~time_bin",
    )
    return model.fit(reml=False)


def _interaction_terms(params_index, treatment_col):
    """
    Robustly find treatment x time_bin interaction parameters in a
    statsmodels result. We do NOT rely on a specific naming pattern,
    because statsmodels can spell the same term several ways depending on
    how patsy processes the formula (e.g. 'C(Treatment)[T.GH CR]:time_bin',
    'time_bin:C(Treatment)[T.GH CR]', 'C(Treatment)[T.GH CR]:time_bin[1.0]',
    etc.). Instead we require the term to contain BOTH the treatment
    factor substring AND 'time_bin'.
    """
    treatment_token = f"C({treatment_col})"
    hits = [
        t for t in params_index
        if (treatment_token in t)
        and ("time_bin" in t)
        and (":" in t)
    ]
    return hits


def _simple_slopes_gee(df, outcome, treatment_col, groups_col, cov_struct_name,
                       family, ref_group=None):
    """
    Re-fit the GEE once per treatment level as reference, so we can read off
    the within-group slope (coefficient on time_bin) for each group.
    Returns a DataFrame with one row per group: slope, robust SE, p, CI.
    """
    levels = sorted(df[treatment_col].dropna().unique().tolist())
    rows = []
    for ref in levels:
        df2 = df.copy()
        df2['_grp'] = np.where(df2[treatment_col] == ref, 'ref', 'other')
        formula = f"{outcome} ~ time_bin * C(_grp)"
        res = _fit_gee(df2, formula, groups_col, cov_struct_name, family)
        term = 'time_bin'
        if term in res.params.index:
            beta = res.params[term]
            se = res.bse[term]
            p = res.pvalues[term]
            ci = res.conf_int().loc[term].values
            rows.append({
                'group': ref,
                'within_group_slope': beta,
                'robust_se': se,
                'ci_low': ci[0],
                'ci_high': ci[1],
                'p_value': p,
            })
    return pd.DataFrame(rows)


def _simple_slopes_mixedlm(df, outcome, treatment_col, groups_col):
    """
    Same idea as the GEE version but for MixedLM: re-fit once per reference
    level and read off the time_bin coefficient within each group.
    """
    levels = sorted(df[treatment_col].dropna().unique().tolist())
    rows = []
    for ref in levels:
        df2 = df.copy()
        df2['_grp'] = np.where(df2[treatment_col] == ref, 'ref', 'other')
        formula = f"{outcome} ~ time_bin * C(_grp)"
        res = _fit_mixedlm(df2, formula, groups_col)
        term = 'time_bin'
        if term in res.params.index:
            beta = res.params[term]
            se = res.bse[term]
            p = res.pvalues[term]
            ci = res.conf_int().loc[term].values
            rows.append({
                'group': ref,
                'within_group_slope': beta,
                'se': se,
                'ci_low': ci[0],
                'ci_high': ci[1],
                'p_value': p,
            })
    return pd.DataFrame(rows)


def analyze_treatment_trends(time_course_df, treatment_col,
                             outcomes=OUTCOMES_FOR_TREND,
                             working_corr=WORKING_CORR,
                             family=FAMILY,
                             reference_group=REFERENCE_GROUP,
                             apply_holm=APPLY_HOLM,
                             small_sample_t=SMALL_SAMPLE_T_CORRECTION):
    """
    For each outcome, fit:
      (a) a GEE with robust SEs and AR(1) working correlation,
          formula: outcome ~ time_bin * C(treatment)
      (b) a linear mixed model with random intercept + random slope per animal
          (sensitivity check at ~12 animals/group)

    Report:
      - the treatment x time interaction coefficients (one per non-reference
        level), with robust SE, CI, z/t p-value
      - a global Wald test across all interaction terms (all interactions
        jointly zero)
      - simple slopes within each group (does each group trend on its own?)
      - Holm-adjusted p-values across the interaction contrasts within each
        outcome (controls family-wise error across the 3 contrasts)
      - the mixed-model equivalent for comparison

    REFERENCE_GROUP (default "GH AL") is fixed as the control. All interaction
    coefficients are contrasts of each other group *against GH AL*.

    Writes a long-format CSV: one row per (outcome, model, term), with
    metadata columns.
    """
    print("\n" + "=" * 70)
    print("TREATMENT x TIME TREND TESTS")
    print(f"Reference/control group: {reference_group}")
    print("=" * 70)

    df = time_course_df.copy()

    if treatment_col not in df.columns:
        raise ValueError(f"treatment_col '{treatment_col}' not in time-course df. "
                         f"Available: {list(df.columns)}")

    # Ensure treatment is categorical (object / category) so C(...) works
    df[treatment_col] = df[treatment_col].astype('category')

    # Force GH AL as the reference level
    if reference_group is not None:
        if reference_group not in df[treatment_col].cat.categories:
            raise ValueError(
                f"reference_group '{reference_group}' not among levels "
                f"{list(df[treatment_col].cat.categories)}. "
                f"Check the spelling / casing in the master sheet."
            )
        cats = [reference_group] + [c for c in df[treatment_col].cat.categories
                                    if c != reference_group]
        df[treatment_col] = df[treatment_col].cat.reorder_categories(cats)

    print(f"Treatment column: '{treatment_col}'")
    print(f"Levels: {list(df[treatment_col].cat.categories)}")
    print(f"Reference (baseline) level: {df[treatment_col].cat.categories[0]}")
    print(f"n animals: {df['video'].nunique()},  n time bins: {df['time_bin'].nunique()}")

    all_rows = []

    for outcome in outcomes:
        sub = df[['video', 'time_bin', treatment_col, outcome]].dropna(subset=[outcome]).copy()
        n_clusters = sub['video'].nunique()
        print(f"\n--- Outcome: {outcome}  (n rows = {len(sub)}, "
              f"n clusters = {n_clusters}) ---")

        if n_clusters < 5:
            print(f"  [SKIP] Only {n_clusters} clusters; too few to fit GEE reliably.")
            continue

        formula = f"{outcome} ~ time_bin * C({treatment_col})"

        # ---------------- GEE ----------------
        try:
            gee_res = _fit_gee(sub, formula, 'video', working_corr, family)
        except Exception as e:
            print(f"  [ERROR] GEE fit failed: {e}")
            continue

        n_params = len(gee_res.params)
        df_resid = max(1, n_clusters - n_params)
        use_t = bool(small_sample_t)

        int_terms = _interaction_terms(gee_res.params.index, treatment_col)
        if not int_terms:
            print("  [WARN] No interaction terms found in GEE result. "
                  "Check that time_bin is numeric and treatment is categorical.")
            continue

        # Collect interaction coefficients
        int_rows = []
        for term in int_terms:
            beta = float(gee_res.params[term])
            se = float(gee_res.bse[term])
            if use_t:
                t_stat = beta / se
                p = 2 * stats.t.sf(abs(t_stat), df=df_resid)
                stat_name = f"t(df={df_resid})"
                stat_val = t_stat
            else:
                z_stat = beta / se
                p = 2 * stats.norm.sf(abs(z_stat))
                stat_name = "z"
                stat_val = z_stat
            ci = gee_res.conf_int().loc[term].values
            int_rows.append({
                'term': term,
                'beta': beta,
                'se': se,
                'ci_low': float(ci[0]),
                'ci_high': float(ci[1]),
                'stat_name': stat_name,
                'stat_value': float(stat_val),
                'p_value': float(p),
                'p_value_holm': np.nan,  # filled below
            })

        # Holm correction across the interaction contrasts within this outcome
        if apply_holm and len(int_rows) > 1:
            raw = [r['p_value'] for r in int_rows]
            adj = _holm_correction(raw)
            for r, a in zip(int_rows, adj):
                r['p_value_holm'] = float(a)

        # Global Wald test: all interaction terms jointly zero
        try:
            R = np.zeros((len(int_terms), len(gee_res.params)))
            for i, term in enumerate(int_terms):
                j = list(gee_res.params.index).index(term)
                R[i, j] = 1.0
            wald = gee_res.wald_test(R, scalar=True)
            global_stat = float(wald.statistic)
            global_p = float(wald.pvalue)
            global_df = len(int_terms)
            print(f"  GEE global Wald test (all {global_df} interactions = 0): "
                  f"chi2({global_df}) = {global_stat:.3f}, p = {global_p:.3g}")
        except Exception as e:
            global_stat, global_p, global_df = np.nan, np.nan, len(int_terms)
            print(f"  [WARN] Global Wald test failed: {e}")

        # Report interaction coefficients
        print(f"  GEE ({working_corr} working corr, robust SEs, "
              f"{'t' if use_t else 'z'}-based p) -- each contrast vs {reference_group}:")
        for r in int_rows:
            holm_str = (f", p_Holm = {r['p_value_holm']:.3g}"
                        if not np.isnan(r['p_value_holm']) else "")
            print(f"    {r['term']}: β = {r['beta']:.4f}, "
                  f"SE = {r['se']:.4f}, "
                  f"95% CI [{r['ci_low']:.4f}, {r['ci_high']:.4f}], "
                  f"p = {r['p_value']:.3g}{holm_str}")

        # Simple slopes within each group (GEE)
        try:
            ss_gee = _simple_slopes_gee(sub, outcome, treatment_col, 'video',
                                        working_corr, family)
            print(f"  GEE simple slopes within each group (cm/s per bin):")
            for _, row in ss_gee.iterrows():
                print(f"    {row['group']}: slope = {row['within_group_slope']:.4f} "
                      f"[{row['ci_low']:.4f}, {row['ci_high']:.4f}], "
                      f"p = {row['p_value']:.3g}")
        except Exception as e:
            print(f"  [WARN] GEE simple slopes failed: {e}")
            ss_gee = pd.DataFrame()

        # ---------------- Mixed model (sensitivity) ----------------
        try:
            mm_res = _fit_mixedlm(sub, formula, 'video')
            mm_int_terms = _interaction_terms(mm_res.params.index, treatment_col)
            print(f"  MixedLM (random intercept + random slope per animal) -- "
                  f"each contrast vs {reference_group}:")
            for term in mm_int_terms:
                beta = float(mm_res.params[term])
                se = float(mm_res.bse[term])
                p = float(mm_res.pvalues[term])
                ci = mm_res.conf_int().loc[term].values
                print(f"    {term}: β = {beta:.4f}, SE = {se:.4f}, "
                      f"95% CI [{ci[0]:.4f}, {ci[1]:.4f}], p = {p:.3g}")
        except Exception as e:
            print(f"  [WARN] MixedLM fit failed: {e}")
            mm_res = None
            mm_int_terms = []

        # Simple slopes within each group (MixedLM)
        try:
            ss_mm = _simple_slopes_mixedlm(sub, outcome, treatment_col, 'video')
            print(f"  MixedLM simple slopes within each group (cm/s per bin):")
            for _, row in ss_mm.iterrows():
                print(f"    {row['group']}: slope = {row['within_group_slope']:.4f} "
                      f"[{row['ci_low']:.4f}, {row['ci_high']:.4f}], "
                      f"p = {row['p_value']:.3g}")
        except Exception as e:
            print(f"  [WARN] MixedLM simple slopes failed: {e}")
            ss_mm = pd.DataFrame()

        # ---------------- Pack results into long-format rows ----------------
        for r in int_rows:
            all_rows.append({
                'outcome': outcome,
                'model': f'GEE ({working_corr}, robust)',
                'term_type': 'interaction',
                'term': r['term'],
                'beta': r['beta'],
                'se': r['se'],
                'ci_low': r['ci_low'],
                'ci_high': r['ci_high'],
                'stat_name': r['stat_name'],
                'stat_value': r['stat_value'],
                'p_value': r['p_value'],
                'p_value_holm': r['p_value_holm'],
                'n_clusters': n_clusters,
                'n_obs': len(sub),
                'global_wald_stat': global_stat,
                'global_wald_df': global_df,
                'global_wald_p': global_p,
                'reference_group': reference_group,
            })

        if mm_res is not None:
            for term in mm_int_terms:
                beta = float(mm_res.params[term])
                se = float(mm_res.bse[term])
                p = float(mm_res.pvalues[term])
                ci = mm_res.conf_int().loc[term].values
                all_rows.append({
                    'outcome': outcome,
                    'model': 'MixedLM (RI+RS)',
                    'term_type': 'interaction',
                    'term': term,
                    'beta': beta,
                    'se': se,
                    'ci_low': float(ci[0]),
                    'ci_high': float(ci[1]),
                    'stat_name': 'z',
                    'stat_value': beta / se,
                    'p_value': p,
                    'p_value_holm': np.nan,
                    'n_clusters': n_clusters,
                    'n_obs': len(sub),
                    'global_wald_stat': np.nan,
                    'global_wald_df': np.nan,
                    'global_wald_p': np.nan,
                    'reference_group': reference_group,
                })

        if not ss_gee.empty:
            for _, row in ss_gee.iterrows():
                all_rows.append({
                    'outcome': outcome,
                    'model': f'GEE ({working_corr}, robust)',
                    'term_type': 'simple_slope',
                    'term': row['group'],
                    'beta': row['within_group_slope'],
                    'se': row['robust_se'],
                    'ci_low': row['ci_low'],
                    'ci_high': row['ci_high'],
                    'stat_name': 'z',
                    'stat_value': row['within_group_slope'] / row['robust_se'],
                    'p_value': row['p_value'],
                    'p_value_holm': np.nan,
                    'n_clusters': n_clusters,
                    'n_obs': len(sub),
                    'global_wald_stat': global_stat,
                    'global_wald_df': global_df,
                    'global_wald_p': global_p,
                    'reference_group': reference_group,
                })

        if not ss_mm.empty:
            for _, row in ss_mm.iterrows():
                all_rows.append({
                    'outcome': outcome,
                    'model': 'MixedLM (RI+RS)',
                    'term_type': 'simple_slope',
                    'term': row['group'],
                    'beta': row['within_group_slope'],
                    'se': row['se'],
                    'ci_low': row['ci_low'],
                    'ci_high': row['ci_high'],
                    'stat_name': 'z',
                    'stat_value': row['within_group_slope'] / row['se'],
                    'p_value': row['p_value'],
                    'p_value_holm': np.nan,
                    'n_clusters': n_clusters,
                    'n_obs': len(sub),
                    'global_wald_stat': np.nan,
                    'global_wald_df': np.nan,
                    'global_wald_p': np.nan,
                    'reference_group': reference_group,
                })

    if not all_rows:
        print("\n[WARN] No treatment-trend results were produced.")
        return pd.DataFrame()

    out_df = pd.DataFrame(all_rows)

    # Save
    out_path = os.path.join(OUTPUT_FOLDER, 'time_course_speed_treatment_trends.csv')
    out_df.to_csv(out_path, index=False)
    print(f"\n✅ Treatment-trend results saved to {out_path}")

    return out_df


# =============================================================================
# Main
# =============================================================================
def build_speed_time_course(aug_folder=AUGMENTED_CSV_FOLDER, master_path=MASTER_SHEET_PATH,
                             output_folder=OUTPUT_FOLDER, fps=FPS,
                             approach_threshold=APPROACH_THRESHOLD,
                             min_speed_for_approach=MIN_SPEED_FOR_APPROACH,
                             interaction_radius_cm=INTERACTION_RADIUS_CM,
                             distance_smooth_window=DISTANCE_SMOOTH_WINDOW,
                             angle_threshold_deg=ANGLE_THRESHOLD_DEG,
                             bin_seconds=TIME_BIN_SECONDS):
    os.makedirs(output_folder, exist_ok=True)

    master, GROUP_COLS = load_master_sheet(master_path)
    master_videos = set(master['video'])

    all_csv = glob(os.path.join(aug_folder, '*.csv'))
    valid_csv = []
    for f in all_csv:
        try:
            df_sample = pd.read_csv(f, nrows=5)
            if 'dist_to_SS_cm' in df_sample.columns and 'abs_angle_to_SS' in df_sample.columns:
                valid_csv.append(f)
        except Exception:
            continue

    video_files = {}
    for f in valid_csv:
        basename = get_base_name_from_file(f)
        video_name = extract_video_name_from_filename(basename, master_videos)
        if video_name is None:
            match = re.search(r'(MVI_\d+_roi\d+)', basename)
            video_name = match.group(1) if match else None
        if video_name is None:
            print(f"Warning: Could not extract video name from {basename}. Skipping.")
            continue
        if video_name in video_files:
            print(f"Warning: duplicate video name '{video_name}' found in {f} and {video_files[video_name]}. Skipping {f}.")
            continue
        video_files[video_name] = f

    if not video_files:
        print(f"No valid video files found in {aug_folder}")
        return None

    print(f"Found {len(video_files)} unique videos. Interaction radius = {interaction_radius_cm} cm, "
          f"facing angle threshold = {angle_threshold_deg}°.")

    time_course_rows = []
    for vname, csv_path in video_files.items():
        print(f"Processing {vname} ...")
        df = pd.read_csv(csv_path)

        masks = compute_speed_masks(df, fps=fps, threshold=approach_threshold,
                                     min_speed_for_approach=min_speed_for_approach,
                                     interaction_radius_cm=interaction_radius_cm,
                                     distance_smooth_window=distance_smooth_window,
                                     angle_threshold_deg=angle_threshold_deg)
        if masks is None:
            print(f"  [SKIP] Missing required columns in {csv_path}.")
            continue

        if vname in master_videos:
            row_match = master.loc[master['video'] == vname].iloc[0]
            group_values = {gc: (row_match[gc] if pd.notna(row_match[gc]) else 'unknown') for gc in GROUP_COLS}
        else:
            print(f"  WARNING: {vname} not in master sheet. Grouping variables set to 'unknown'.")
            group_values = {gc: 'unknown' for gc in GROUP_COLS}

        tc = compute_speed_time_course(masks, vname, fps=fps, bin_seconds=bin_seconds)
        for gc in GROUP_COLS:
            tc[gc] = group_values[gc]
        time_course_rows.append(tc)

    if not time_course_rows:
        print("No videos produced usable data.")
        return None

    time_course_df = pd.concat(time_course_rows, ignore_index=True)
    out_path = os.path.join(output_folder, 'time_course_speed.csv')
    time_course_df.to_csv(out_path, index=False)
    print(f"\n✅ Speed time-course data saved to {out_path}")

    # ---- Per-video fit quality ----
    fit_df = compute_fit_quality(time_course_df, GROUP_COLS)
    fit_path = os.path.join(output_folder, 'time_course_speed_fit_quality.csv')
    fit_df.to_csv(fit_path, index=False)
    print(f"✅ Per-video fit-quality metrics (slope, R^2, p) saved to {fit_path}")
    print_fit_summary(fit_df)

    # ---- Treatment x time trends ----
    if TREATMENT_COL is not None and TREATMENT_COL in time_course_df.columns:
        analyze_treatment_trends(
            time_course_df,
            treatment_col=TREATMENT_COL,
            outcomes=OUTCOMES_FOR_TREND,
            working_corr=WORKING_CORR,
            family=FAMILY,
            reference_group=REFERENCE_GROUP,
            apply_holm=APPLY_HOLM,
            small_sample_t=SMALL_SAMPLE_T_CORRECTION,
        )
    else:
        print(f"\n[INFO] Skipping treatment-trend stage: "
              f"TREATMENT_COL='{TREATMENT_COL}' not in time-course df.")

    return time_course_df


if __name__ == "__main__":
    build_speed_time_course(
        aug_folder=AUGMENTED_CSV_FOLDER,
        master_path=MASTER_SHEET_PATH,
        output_folder=OUTPUT_FOLDER,
        fps=FPS,
        approach_threshold=APPROACH_THRESHOLD,
        min_speed_for_approach=MIN_SPEED_FOR_APPROACH,
        interaction_radius_cm=INTERACTION_RADIUS_CM,
        distance_smooth_window=DISTANCE_SMOOTH_WINDOW,
        angle_threshold_deg=ANGLE_THRESHOLD_DEG,
        bin_seconds=TIME_BIN_SECONDS,
    )

  Grouping variables that will be tested: ['Treatment', 'Housing', 'Food']
Found 49 unique videos. Interaction radius = 6.0 cm, facing angle threshold = 45.0°.
Processing MVI_0614_roi2 ...
Processing MVI_0091_roi1 ...
Processing MVI_0616_roi1 ...
Processing MVI_0075_roi1 ...
Processing MVI_0319_roi2 ...
Processing MVI_0083_roi1 ...
Processing MVI_0081_roi2 ...
Processing MVI_0311_roi1 ...
Processing MVI_0322_roi1 ...
Processing MVI_0006_roi2 ...
Processing MVI_0074_roi1 ...
Processing MVI_0617_roi1 ...
Processing MVI_0611_roi1 ...
Processing MVI_0618_roi2 ...
Processing MVI_0010_roi1 ...
Processing MVI_0086_roi2 ...
Processing MVI_0327_roi2 ...
Processing MVI_0011_roi1 ...
Processing MVI_0325_roi1 ...
Processing MVI_0314_roi2 ...
Processing MVI_0013_roi2 ...
Processing MVI_0612_roi2 ...
Processing MVI_0008_roi1 ...
Processing MVI_0078_roi1 ...
Processing MVI_0010_roi2 ...
Processing MVI_0086_roi1 ...
Processing MVI_0611_roi2 ...
Processing MVI_0010_roi2copy ...
Processing MVI_0618_roi1

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


  GEE global Wald test (all 3 interactions = 0): chi2(3) = 3.519, p = 0.318
  GEE (ar working corr, robust SEs, t-based p) -- each contrast vs GH AL:
    time_bin:C(Treatment)[T.GH FD]: β = 0.0758, SE = 0.2474, 95% CI [-0.4092, 0.5608], p = 0.761, p_Holm = 1
    time_bin:C(Treatment)[T.SH AL]: β = 0.0275, SE = 0.1989, 95% CI [-0.3624, 0.4173], p = 0.891, p_Holm = 1
    time_bin:C(Treatment)[T.SH FD]: β = -0.3714, SE = 0.2175, 95% CI [-0.7976, 0.0549], p = 0.0953, p_Holm = 0.286


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


  GEE simple slopes within each group (cm/s per bin):
    GH AL: slope = -0.0426 [-0.2799, 0.1946], p = 0.725
    GH FD: slope = -0.0515 [-0.2625, 0.1595], p = 0.632
    SH AL: slope = -0.0559 [-0.2828, 0.1710], p = 0.629
    SH FD: slope = 0.1346 [-0.0551, 0.3244], p = 0.164


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2200: Convergence

... (5 outputs left out)

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(


  MixedLM (random intercept + random slope per animal) -- each contrast vs GH AL:
    time_bin:C(Treatment)[T.GH FD]: β = 0.0874, SE = 0.3778, 95% CI [-0.6531, 0.8279], p = 0.817
    time_bin:C(Treatment)[T.SH AL]: β = -0.0867, SE = 0.3597, 95% CI [-0.7917, 0.6183], p = 0.81
    time_bin:C(Treatment)[T.SH FD]: β = 0.6078, SE = 0.3374, 95% CI [-0.0535, 1.2691], p = 0.0717


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals


  MixedLM simple slopes within each group (cm/s per bin):
    GH AL: slope = 0.1879 [-0.0833, 0.4591], p = 0.174
    GH FD: slope = 0.1613 [-0.1065, 0.4290], p = 0.238
    SH AL: slope = 0.2085 [-0.0650, 0.4819], p = 0.135
    SH FD: slope = -0.0826 [-0.3706, 0.2054], p = 0.574

✅ Treatment-trend results saved to advanced_analysis_plots/time_course_speed_treatment_trends.csv


In [ ]:

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

# =============================================================================
# CONFIG
# =============================================================================
OUTPUT_FOLDER = "advanced_analysis_plots"
TIME_COURSE_CSV = os.path.join(OUTPUT_FOLDER, "time_course_speed.csv")
TRENDS_CSV = os.path.join(OUTPUT_FOLDER, "time_course_speed_treatment_trends.csv")
TREATMENT_COL = "Treatment"
REFERENCE_GROUP = "GH AL"
OUTCOMES = ["approach_speed_SS", "approach_speed_FS"]
OUTCOME_LABELS = {
    "approach_speed_SS": "Approach speed toward social stimulus (SS)",
    "approach_speed_FS": "Approach speed toward food stimulus (FS)",
}
BIN_SECONDS = 60          # 1 bin = 1 minute
X_MAX_MINUTES = 10        # x-axis runs 0 -> 10 min
Y_MIN_CMS = 5             # y-axis floor (truncated, note in caption)
Y_MAX_CMS = 30            # y-axis top, shared by SS and FS panels

# --- Font: Arial, size 12, bold ---
FONT_FAMILY = "Arial"
FONT_SIZE = 12
FONT_WEIGHT = "bold"

# -----------------------------------------------------------------------------
# Colour scheme: MATCHES THE PUBLISHED FIGURE.
# -----------------------------------------------------------------------------
GROUP_COLORS = {
    "GH AL": "#D55E00",   # dark orange-red   (reference group)
    "GH FD": "#E69F00",   # light orange
    "SH AL": "#155D86",   # dark blue
    "SH FD": "#39CECE",   # light aqua
}
DEFAULT_COLOR = "#7F7F7F"

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,

    # Font: Arial, 12 pt, bold everywhere
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_FAMILY, "Helvetica", "DejaVu Sans"],
    "font.size": FONT_SIZE,
    "font.weight": FONT_WEIGHT,
    "axes.titlesize": FONT_SIZE,
    "axes.titleweight": FONT_WEIGHT,
    "axes.labelsize": FONT_SIZE,
    "axes.labelweight": FONT_WEIGHT,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "figure.titlesize": FONT_SIZE,
    "figure.titleweight": FONT_WEIGHT,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    "axes.spines.top": False,
    "axes.spines.right": False,
})


# =============================================================================
# Helpers
# =============================================================================
def _color_for(group):
    return GROUP_COLORS.get(group, DEFAULT_COLOR)


def _save(fig, name):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_FOLDER, f"{name}.{ext}")
        fig.savefig(path, bbox_inches="tight")
    print(f"  saved {name}.png / .pdf")


def _order_groups(groups):
    """Reference group first, then alphabetical."""
    groups = sorted(groups)
    if REFERENCE_GROUP in groups:
        groups = [REFERENCE_GROUP] + [g for g in groups if g != REFERENCE_GROUP]
    return groups


def _hyphenated_minute_ticks(ax, tmax_minutes=10, step=1):
    """
    Label x ticks as '0-1', '1-2', ... '9-10' so the reader sees each tick
    is the centre of a 1-minute bin. Guarantees the final label (tmax) is
    present and not clipped.
    """
    dash = "\u2013"   # en-dash
    ticks = np.arange(0, tmax_minutes, step)
    labels = [f"{int(t)}{dash}{int(t + step)}" for t in ticks]

    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    # Small right-hand padding so the '9-10' label isn't clipped
    ax.set_xlim(-0.2, tmax_minutes + 0.2)

    # Minor ticks at bin boundaries so the span is visible
    ax.set_xticks(np.arange(0.5, tmax_minutes, step), minor=True)
    ax.tick_params(axis="x", which="minor", length=3, color="0.5")

    # Make sure matplotlib doesn't quietly drop the endpoint label
    for lbl in ax.get_xticklabels():
        lbl.set_clip_on(False)


# =============================================================================
# FIGURE 1 -- model-fitted per-group time course with model-based CI
# =============================================================================
def _fit_group_curve(df, outcome, treatment_col, family=None,
                     working_corr="ar"):
    """
    Fit the same GEE as the analysis stage and return, per group, the fitted
    marginal mean at every time bin plus a 95% CI ribbon.

    Returns a DataFrame with columns:
        group, time_bin, fitted, ci_low, ci_high
    """
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    if family is None:
        family = sm.families.Gaussian()

    cov_map = {
        "ar": sm.cov_struct.Autoregressive,
        "exchangeable": sm.cov_struct.Exchangeable,
        "independence": sm.cov_struct.Independence,
    }

    sub = df[["video", "time_bin", treatment_col, outcome]].dropna(subset=[outcome]).copy()
    sub[treatment_col] = sub[treatment_col].astype("category")

    formula = f"{outcome} ~ time_bin * C({treatment_col})"
    model = smf.gee(
        formula=formula,
        groups="video",
        data=sub,
        family=family,
        cov_struct=cov_map[working_corr](),
    )
    res = model.fit(cov_type="robust")

    # Prediction grid: every time bin, every group
    bins = np.sort(sub["time_bin"].unique())
    groups = sorted(sub[treatment_col].cat.categories.tolist())

    pred_rows = []
    for grp in groups:
        for b in bins:
            pred_rows.append({"time_bin": b, treatment_col: grp})
    pred_df = pd.DataFrame(pred_rows)
    pred_df[treatment_col] = pd.Categorical(
        pred_df[treatment_col], categories=sub[treatment_col].cat.categories
    )

    pred = res.predict(pred_df)

    # CI on the linear predictor
    from patsy import dmatrix
    design_info = res.model.data.design_info
    X = dmatrix(design_info, pred_df, return_type="dataframe")
    X = X[res.params.index]
    Xv = X.values
    vcov = res.cov_params().values
    se = np.sqrt(np.einsum("ij,jk,ik->i", Xv, vcov, Xv))
    ci_low = pred - 1.96 * se
    ci_high = pred + 1.96 * se

    out = pd.DataFrame({
        "group": pred_df[treatment_col].astype(str).values,
        "time_bin": pred_df["time_bin"].values,
        "fitted": pred.values,
        "ci_low": ci_low,
        "ci_high": ci_high,
    })
    return out


def plot_timecourse_ribbons(df):
    """
    Model-fitted per-group trajectories with model-based 95% CI ribbons.
    Raw bin means drawn as faint dots. Y-axis shared and truncated to
    5-25 cm/s; x-axis runs 0-10 min with hyphenated bin labels.
    """
    groups = _order_groups(df[TREATMENT_COL].dropna().unique().tolist())

    fig, axes = plt.subplots(
        1, 2, figsize=(11, 4.2), sharey=True
    )
    for ax, outcome in zip(axes, OUTCOMES):
        fitted = _fit_group_curve(df, outcome, TREATMENT_COL)

        for grp in groups:
            fg = fitted[fitted["group"] == grp].sort_values("time_bin")
            if fg.empty:
                continue
            x = fg["time_bin"] * (BIN_SECONDS / 60.0)
            color = _color_for(grp)
            lw = 2.4 if grp == REFERENCE_GROUP else 1.8

            ax.plot(x, fg["fitted"], color=color, lw=lw, label=grp, zorder=3)
            ax.fill_between(x, fg["ci_low"], fg["ci_high"],
                            color=color, alpha=0.18, lw=0, zorder=2)

            raw = (df[df[TREATMENT_COL] == grp]
                   .groupby("time_bin")[outcome].mean().reset_index())
            xr = raw["time_bin"] * (BIN_SECONDS / 60.0)
            ax.scatter(xr, raw[outcome], color=color, s=14, alpha=0.55,
                       edgecolor="white", linewidth=0.5, zorder=4)

        ax.set_xlabel("Time in session (minutes)")
        ax.set_title(OUTCOME_LABELS[outcome])
        ax.set_ylim(Y_MIN_CMS, Y_MAX_CMS)
        ax.tick_params(axis="y", labelleft=True)
        _hyphenated_minute_ticks(ax, X_MAX_MINUTES)

        # Warn if any fitted CI or raw value is clipped by the y-limits
        if pd.notna(fitted["ci_low"].min()) and fitted["ci_low"].min() < Y_MIN_CMS:
            print(f"  [WARN] {outcome}: fitted CI lower bound "
                  f"{fitted['ci_low'].min():.2f} < Y_MIN_CMS = {Y_MIN_CMS}; "
                  f"ribbon will be clipped at the bottom.")
        if pd.notna(fitted["ci_high"].max()) and fitted["ci_high"].max() > Y_MAX_CMS:
            print(f"  [WARN] {outcome}: fitted CI upper bound "
                  f"{fitted['ci_high'].max():.2f} > Y_MAX_CMS = {Y_MAX_CMS}; "
                  f"ribbon will be clipped at the top.")
        if pd.notna(df[outcome].min()) and df[outcome].min() < Y_MIN_CMS:
            print(f"  [WARN] {outcome}: raw min {df[outcome].min():.2f} "
                  f"< Y_MIN_CMS = {Y_MIN_CMS}; some points will be clipped.")
        if pd.notna(df[outcome].max()) and df[outcome].max() > Y_MAX_CMS:
            print(f"  [WARN] {outcome}: raw max {df[outcome].max():.2f} "
                  f"> Y_MAX_CMS = {Y_MAX_CMS}; some points will be clipped.")

    # Only the leftmost panel needs a y-label since the axis is shared
    axes[0].set_ylabel("Approach speed (cm/s)")
    axes[1].set_ylabel("")
    axes[0].legend(frameon=False, loc="best")

    fig.suptitle("Model-fitted approach speed over the session, by treatment group "
                 "(GEE fit; ribbon = 95% CI; dots = raw bin means)",
                 y=1.03, fontsize=FONT_SIZE, fontweight=FONT_WEIGHT)
    fig.tight_layout()
    _save(fig, "fig1_timecourse_ribbons")
    plt.close(fig)


# =============================================================================
# FIGURE 2 -- per-animal spaghetti, faceted by group
# =============================================================================
def plot_spaghetti_by_group(df):
    """
    One panel per (outcome, treatment group). Faint lines = one animal each;
    bold line = group mean. Y shared within each outcome row, 5-25 cm/s.
    X runs 0-10 min with hyphenated bin labels on every panel.
    """
    groups = _order_groups(df[TREATMENT_COL].dropna().unique().tolist())
    n_rows = len(OUTCOMES)
    n_cols = len(groups)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.8 * n_rows),
        sharex=True, sharey="row"
    )

    for i, outcome in enumerate(OUTCOMES):
        # Warn once per outcome about clipping
        if pd.notna(df[outcome].min()) and df[outcome].min() < Y_MIN_CMS:
            print(f"  [WARN] {outcome}: raw min {df[outcome].min():.2f} "
                  f"< Y_MIN_CMS = {Y_MIN_CMS}; spaghetti lines will be clipped.")
        if pd.notna(df[outcome].max()) and df[outcome].max() > Y_MAX_CMS:
            print(f"  [WARN] {outcome}: raw max {df[outcome].max():.2f} "
                  f"> Y_MAX_CMS = {Y_MAX_CMS}; spaghetti lines will be clipped.")

        for j, grp in enumerate(groups):
            ax = axes[i, j]
            g = df[df[TREATMENT_COL] == grp]

            for vid, ga in g.groupby("video"):
                ga_sorted = ga.sort_values("time_bin")
                x = ga_sorted["time_bin"] * (BIN_SECONDS / 60.0)
                y = ga_sorted[outcome]
                ax.plot(x, y, color="0.75", lw=0.6, alpha=0.7, zorder=1)

            mean_by_bin = g.groupby("time_bin")[outcome].mean()
            xm = mean_by_bin.index * (BIN_SECONDS / 60.0)
            ax.plot(xm, mean_by_bin.values, color=_color_for(grp), lw=2.2,
                    marker="o", ms=3, zorder=3)

            if i == 0:
                ax.set_title(grp, color=_color_for(grp))
            if j == 0:
                ax.set_ylabel("Approach speed\n(cm/s)")

            ax.set_ylim(Y_MIN_CMS, Y_MAX_CMS)
            ax.tick_params(axis="y", labelleft=True)
            _hyphenated_minute_ticks(ax, X_MAX_MINUTES)

    # Row labels on the left
    for i, outcome in enumerate(OUTCOMES):
        axes[i, 0].annotate(
            "SS" if "SS" in outcome else "FS",
            xy=(-0.32, 0.5), xycoords="axes fraction",
            rotation=90, va="center", ha="center",
            fontsize=FONT_SIZE, fontweight=FONT_WEIGHT,
        )

    fig.suptitle("Per-animal time courses (grey) with group mean (colored)",
                 y=1.02, fontsize=FONT_SIZE, fontweight=FONT_WEIGHT)
    fig.tight_layout()
    _save(fig, "fig2_spaghetti_by_group")
    plt.close(fig)


# =============================================================================
# FIGURE 3 -- simple-slope forest plot, GEE vs MixedLM
# =============================================================================
def plot_slope_forest(trends_df):
    """
    For each outcome, plot each group's simple slope with 95% CI.
    Two rows per group: GEE (robust) and MixedLM.
    """
    ss = trends_df[trends_df["term_type"] == "simple_slope"].copy()
    if ss.empty:
        print("  [skip] no simple_slope rows found")
        return
    ss = ss.rename(columns={"beta": "slope", "se": "slope_se"})

    groups = _order_groups(ss["term"].unique().tolist())
    models = sorted(ss["model"].unique().tolist())

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=False)
    for ax, outcome in zip(axes, OUTCOMES):
        sub = ss[ss["outcome"] == outcome]
        rows = []
        for g in groups:
            for m in models:
                r = sub[(sub["term"] == g) & (sub["model"] == m)]
                if len(r):
                    rows.append(r.iloc[0])
        if not rows:
            ax.set_visible(False)
            continue

        y_positions = []
        y_labels = []
        y = 0
        for g in groups:
            for m in models:
                r = sub[(sub["term"] == g) & (sub["model"] == m)]
                if len(r):
                    r = r.iloc[0]
                    y_positions.append((y, r))
                    y_labels.append(f"{g}  [{m}]")
                    y += 1
            y += 0.6

        for y, r in y_positions:
            color = _color_for(r["term"])
            is_gee = "GEE" in r["model"]
            ax.errorbar(
                r["slope"], y,
                xerr=[[r["slope"] - r["ci_low"]], [r["ci_high"] - r["slope"]]],
                fmt="o" if is_gee else "s",
                color=color,
                mfc=color if is_gee else "white",
                mec=color,
                ms=6, capsize=3, lw=1.4,
                zorder=3
            )

        ax.axvline(0, color="0.5", lw=0.8, ls="--", zorder=1)
        ax.set_yticks([y for y, _ in y_positions])
        ax.set_yticklabels(y_labels, fontsize=FONT_SIZE)
        ax.invert_yaxis()
        ax.set_xlabel("Slope (cm/s per minute)")
        ax.set_title(OUTCOME_LABELS[outcome])

    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], marker="o", color="0.3", lw=0, ms=7, label="GEE (robust)"),
        Line2D([0], [0], marker="s", color="0.3", lw=0, mfc="white", ms=7, label="MixedLM"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=2,
               frameon=False, bbox_to_anchor=(0.5, -0.05))
    fig.suptitle("Within-group simple slopes: does each group trend on its own?",
                 y=1.02, fontsize=FONT_SIZE, fontweight=FONT_WEIGHT)
    fig.tight_layout()
    _save(fig, "fig3_slope_forest")
    plt.close(fig)


# =============================================================================
# FIGURE 4 -- interaction contrasts vs reference (forest)
# =============================================================================
def plot_interaction_forest(trends_df):
    """
    Treatment x time interaction coefficients (each non-reference group vs
    GH AL) with 95% CI, from GEE and MixedLM.
    """
    inter = trends_df[trends_df["term_type"] == "interaction"].copy()
    if inter.empty:
        print("  [skip] no interaction rows found")
        return

    def extract_group(term):
        m = re.search(r"\[T\.(.+?)\]", term)
        return m.group(1) if m else term
    inter["group"] = inter["term"].apply(extract_group)

    groups = _order_groups(inter["group"].unique().tolist())
    groups = [g for g in groups if g != REFERENCE_GROUP]
    models = sorted(inter["model"].unique().tolist())

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.0), sharey=False)
    for ax, outcome in zip(axes, OUTCOMES):
        sub = inter[inter["outcome"] == outcome]
        if sub.empty:
            ax.set_visible(False)
            continue

        y_positions = []
        y = 0
        for g in groups:
            for m in models:
                r = sub[(sub["group"] == g) & (sub["model"] == m)]
                if len(r):
                    y_positions.append((y, r.iloc[0]))
                    y += 1
            y += 0.5

        for y, r in y_positions:
            is_gee = "GEE" in r["model"]
            color = _color_for(r["group"])

            ax.errorbar(
                r["beta"], y,
                xerr=[[r["beta"] - r["ci_low"]], [r["ci_high"] - r["beta"]]],
                fmt="o" if is_gee else "s",
                color=color,
                mfc=color if is_gee else "white",
                mec=color,
                ms=6, capsize=3, lw=1.4,
                zorder=3
            )

            if is_gee:
                p_holm = r.get("p_value_holm", np.nan)
                if pd.notna(p_holm):
                    label = f"p_Holm = {p_holm:.3f}"
                else:
                    label = f"p = {r['p_value']:.3f}"
            else:
                label = f"p_raw = {r['p_value']:.3f}"

            ax.text(r["ci_high"] + 0.04, y, label,
                    va="center", ha="left",
                    fontsize=9, color="0.25")

        ax.axvline(0, color="0.5", lw=0.8, ls="--", zorder=1)
        yticks = [y for y, _ in y_positions]
        ylabels = []
        for y, r in y_positions:
            tag = "GEE" if "GEE" in r["model"] else "MixedLM"
            ylabels.append(f"{r['group']}  [{tag}]")
        ax.set_yticks(yticks)
        ax.set_yticklabels(ylabels, fontsize=FONT_SIZE)
        ax.invert_yaxis()
        ax.set_xlabel(f"Difference in slope vs {REFERENCE_GROUP} (cm/s per minute)")

        gwp = sub["global_wald_p"].dropna()
        title = OUTCOME_LABELS[outcome]
        if len(gwp):
            title += f"\n(global Wald p = {gwp.iloc[0]:.3f})"
        ax.set_title(title, fontsize=FONT_SIZE, fontweight=FONT_WEIGHT)

        xlim = ax.get_xlim()
        ax.set_xlim(xlim[0], xlim[1] + 0.35 * (xlim[1] - xlim[0]))

    fig.suptitle(
        f"Treatment x time interactions vs {REFERENCE_GROUP}\n"
        "GEE p-values are Holm-corrected across the 3 contrasts; "
        "MixedLM p-values are raw.",
        y=1.10, fontsize=FONT_SIZE, fontweight=FONT_WEIGHT
    )
    fig.tight_layout()
    _save(fig, "fig4_interaction_forest")
    plt.close(fig)


# =============================================================================
# Main
# =============================================================================
def main():
    if not os.path.exists(TIME_COURSE_CSV):
        raise FileNotFoundError(
            f"Couldn't find {TIME_COURSE_CSV}. Run the analysis script first."
        )
    if not os.path.exists(TRENDS_CSV):
        raise FileNotFoundError(
            f"Couldn't find {TRENDS_CSV}. Run the analysis script first."
        )

    df = pd.read_csv(TIME_COURSE_CSV)
    trends = pd.read_csv(TRENDS_CSV)

    assert TREATMENT_COL in df.columns, f"{TREATMENT_COL} not in time_course_speed.csv"
    groups_present = sorted(df[TREATMENT_COL].dropna().unique().tolist())
    print(f"Groups found: {groups_present}")
    if REFERENCE_GROUP not in groups_present:
        print(f"  [WARN] reference group '{REFERENCE_GROUP}' not in data; "
              f"figures will still render but reference styling won't apply.")

    print("Rendering figures ...")
    plot_timecourse_ribbons(df)
    plot_spaghetti_by_group(df)
    plot_slope_forest(trends)
    plot_interaction_forest(trends)
    print("\nDone. Figures are in:", OUTPUT_FOLDER)


if __name__ == "__main__":
    main()

Groups found: ['GH AL', 'GH FD', 'SH AL', 'SH FD']
Rendering figures ...


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


  [WARN] approach_speed_SS: raw min 1.20 < Y_MIN_CMS = 5; some points will be clipped.
  [WARN] approach_speed_SS: raw max 36.81 > Y_MAX_CMS = 30; some points will be clipped.
  [WARN] approach_speed_FS: raw min 1.20 < Y_MIN_CMS = 5; some points will be clipped.
  [WARN] approach_speed_FS: raw max 39.46 > Y_MAX_CMS = 30; some points will be clipped.


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


  saved fig1_timecourse_ribbons.png / .pdf
  [WARN] approach_speed_SS: raw min 1.20 < Y_MIN_CMS = 5; spaghetti lines will be clipped.
  [WARN] approach_speed_SS: raw max 36.81 > Y_MAX_CMS = 30; spaghetti lines will be clipped.
  [WARN] approach_speed_FS: raw min 1.20 < Y_MIN_CMS = 5; spaghetti lines will be clipped.
  [WARN] approach_speed_FS: raw max 39.46 > Y_MAX_CMS = 30; spaghetti lines will be clipped.
  saved fig2_spaghetti_by_group.png / .pdf
  saved fig3_slope_forest.png / .pdf
  saved fig4_interaction_forest.png / .pdf

Done. Figures are in: advanced_analysis_plots
